## Časť 1, Téma 1: Úvod do glitchingu s hodinami

---
POZNÁMKA: Toto cvičenie odkazuje na niektoré (komerčné) školiace materiály na stránke ChipWhisperer.io. Cvičenie môžete voľne spúšťať a používať v súlade s licenciou open source (vrátane použitia vo vašich vlastných kurzoch, ak ich distribuujete podobným spôsobom), musíte však zachovať odkaz na tento zdroj. Zvážte účasť na našom školiacom kurze, aby ste si mohli vychutnať plnohodnotný zážitok.

---

**ZHRNUTIE:** *Mikrokontroléry a FPGA majú rad prevádzkových podmienok, ktoré musia byť splnené, aby zariadenie fungovalo správne. Ak sa tieto podmienky nedodržia, zariadenia začnú vykazovať poruchy, pričom extrémnejšie porušenia môžu viesť k úplnému zastaveniu zariadenia alebo dokonca k jeho poškodeniu. Ak tieto prevádzkové podmienky na veľmi krátku dobu porušíme, môžeme spôsobiť rôzne dočasné chyby.*

*V tomto cvičení sa budeme zaoberať glitchingom hodín, ktorý vkladá krátke glitche do hodín zariadenia. Toto využijeme na to, aby cieľ, ktorý sčítava čísla v slučke, dospel k nesprávnemu výsledku.*

**Čo sa naučíme:**

* Pochopiť účinky glitchingu hodín
* Preskúmať glitch modul ChipWhisperer
* Použiť glitching hodín na narušenie algoritmu cieľa

## Teória porúch hodinového signálu

Digitálne hardvérové zariadenia takmer vždy vyžadujú nejakú formu spoľahlivého hodinového signálu. Môžeme manipulovať s hodinovým signálom dodávaným do zariadenia a vyvolať tak neočakávané správanie. Zameriame sa tu na mikrokontroléry, avšak pomocou tejto techniky je možné vyvolať poruchy aj v iných digitálnych zariadeniach (napr. hardvérových šifrovacích akcelerátoroch).

Najprv sa pozrime na mikrokontrolér. Nasledujúci obrázok je výňatok z dátového listu Atmel AVR ATMega328P:

![A2_1](img/Mcu-unglitched.png)

Namiesto načítania každej inštrukcie z FLASH a vykonania celého procesu má systém "rúry" na urýchlenie procesu vykonávania. To znamená, že inštrukcia sa dekóduje, zatiaľ čo sa načítava ďalšia, ako ukazuje nasledujúci diagram:

![A2_2](img/Clock-normal.png)

Ak však upravíme takt, môže nastať situácia, keď systém nebude mať dostatok času na skutočné vykonanie inštrukcie. Zvážte nasledujúci prípad, kde je inštrukcia Execute #1 v podstate preskočená. Skôr než systém stihne inštrukciu skutočne vykonať, príde ďalší takt, čo spôsobí, že mikrokontrolér začne vykonávať ďalšiu inštrukciu:

![A2_3](img/Clock-glitched.png)

To spôsobí, že mikrokontrolér preskočí inštrukciu. Takéto útoky môžu byť v praxi nesmierne účinné. Zvážte napríklad nasledujúci kód z `linux-util-2.24`:

```C
/*
 *   auth.c -- PAM authorization code, common between chsh and chfn
 *   (c) 2012 by Cody Maloney <cmaloney@theoreticalchaos.com>
 *
 *   this program is free software.  you can redistribute it and
 *   modify it under the terms of the gnu general public license.
 *   there is no warranty.
 *
 */

#include "auth.h"
#include "pamfail.h"

int auth_pam(const char *service_name, uid_t uid, const char *username)
{
    if (uid != 0) {
        pam_handle_t *pamh = NULL;
        struct pam_conv conv = { misc_conv, NULL };
        int retcode;

        retcode = pam_start(service_name, username, &conv, &pamh);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_authenticate(pamh, 0);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_acct_mgmt(pamh, 0);
        if (retcode == PAM_NEW_AUTHTOK_REQD)
            retcode =
                pam_chauthtok(pamh, PAM_CHANGE_EXPIRED_AUTHTOK);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_setcred(pamh, 0);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        pam_end(pamh, 0);
        /* no need to establish a session; this isn't a
         * session-oriented activity...  */
    }
    return TRUE;
}
```

Toto je prihlasovací kód pre operačný systém Linux. Všimnite si, že keby sme mohli preskočiť kontrolu `if (uid != 0)` a jednoducho prejsť na koniec, vyhnuli by sme sa nutnosti zadávať heslo. V tom spočíva sila glitch útokov – nejde o to, že by sme prelomili šifrovanie, ale jednoducho obchádzame celý autentifikačný modul! 

### Glitch hardvér

Systém ChipWhisperer Glitch používa rovnakú synchronnú metodiku ako jeho zachytávanie analýzy bočného kanála (SCA). Na generovanie glitchov sa používa systémové hodiny (ktoré môžu pochádzať buď z ChipWhisperer, alebo z testovaného zariadenia (DUT)). Tieto glitche sa potom vkladajú späť do hodín, hoci je možné použiť glitche samostatne na iné účely (napr. na napäťové glitche, EM glitche).

Generovanie glitchov sa vykonáva pomocou dvoch modulov s variabilným fázovým posunom, nakonfigurovaných nasledovne:

![A2_4](img/Glitchgen-phaseshift.png)

V CW-Husky je jeden dôležitý rozdiel: výstup fázového posunu 1 nie je invertovaný predtým, ako je spojený logickým AND s výstupom fázového posunu 2.

Aktivačná linka sa používa na určenie, kedy sa glitche vkladajú. Glitche sa môžu vkladať nepretržite (užitočné pre vývoj) alebo sa môžu spúšťať nejakou udalosťou. Nasledujúci obrázok ukazuje, ako sa glitch môže multiplexovať na výstup do testovaného zariadenia (DUT).

![A2_5](img/Glitchgen-mux.png)


### Hardware Support: CW-Lite/Pro

Bloky fázového posunu využívajú bloky Digital Clock Manager (DCM) v rámci FPGA. Tieto bloky majú obmedzenú podporu pre konfiguráciu parametrov za behu, ako je fázové oneskorenie a generovanie frekvencie, a pre dosiahnutie maximálneho výkonu musí byť konfigurácia pevne stanovená už vo fáze návrhu. Funkcia nastavenia za behu poskytovaná spoločnosťou Xilinx dokáže fázový posun zmeniť len o približne +/- 5 ns v krokoch po 30 pS (presné hodnoty sa líšia v závislosti od prevádzkových podmienok).

Pre väčšinu prevádzkových podmienok je to nedostatočné – pri útoku na cieľ pri 7,37 MHz by mal taktový cyklus periódu 136 nS. S cieľom poskytnúť väčší rozsah nastavenia sa používa pokročilá funkcia FPGA nazývaná čiastočná rekonfigurácia (PR). Systém PR vyžaduje špeciálne čiastočné bitové toky, ktoré obsahujú úpravy bitového toku FPGA. Tieto sú uložené ako dva súbory vo vnútri zip súboru „firmware“, ktorý obsahuje bitstream FPGA spolu so súborom s názvom `glitchwidth.p` a súborom s názvom `glitchoffset.p`. Ak sa do FPGA načíta samostatný bitstream (t. j. nie zo zip súboru), systém čiastočnej rekonfigurácie je deaktivovaný, pretože načítanie nesprávnych súborov čiastočnej rekonfigurácie by mohlo poškodiť FPGA. Toto poškodenie je väčšinou teoretické, pravdepodobnejšie je, že FPGA nebude fungovať správne.

Ak v priebehu tohto tutoriálu zistíte, že FPGA prestala reagovať (t. j. určité funkcie už nefungujú správne), môže to znamenať, že údaje čiastočnej rekonfigurácie sú nesprávne.

Ako tieto funkcie používať, si ukážeme neskôr v tomto návode.

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
SS_VER = 'SS_VER_2_1'

In [2]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

OSError: Could not find ChipWhisperer. Is it connected?

OSError: Could not find ChipWhisperer. Is it connected?

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
fw_path = "../../firmware/mcu/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER=='SS_VER_2_1':
    target.reset_comms()

Pri pokuse o využitie glitchov pravdepodobne niekoľkokrát spôsobíme pád cieľa. Vytvorte funkciu na obnovenie cieľa:

In [ ]:
if PLATFORM == "CWLITEXMEGA":
    def reboot_flush():            
        scope.io.pdic = False
        time.sleep(0.1)
        scope.io.pdic = "high_z"
        time.sleep(0.1)
        #Flush garbage too
        target.flush()

## Komunikácia

V tomto cvičení predstavíme novú metódu: `target.simpleserial_read_witherrors()`. Očakávame odpoveď v formáte simpleserial; glitch však často spôsobí zlyhanie cieľového zariadenia a vrátenie neplatného reťazca. Táto metóda to všetko vyrieši za nás. Okrem toho nám oznámi, či bola odpoveď platná a aký bol kód chyby. Použite ju takto:

In [ ]:
#Do glitch loop
target.simpleserial_write("g", bytearray([]))

val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

#print(bytearray(val['full_response'].encode('latin-1')))
print(val)

## Cieľový firmvér

V rámci tohto cvičenia je naším cieľom dosiahnuť, aby nasledujúci kód vygeneroval nesprávny výsledok:

```C
uint8_t glitch_loop(uint8_t* in)
{
    volatile uint16_t i, j;
    volatile uint32_t cnt;
    cnt = 0;
    trigger_high();
    for(i=0; i<50; i++){
        for(j=0; j<50; j++){
            cnt++;
        }
    }
    trigger_low();
    simpleserial_put('r', 4, (uint8_t*)&cnt);
    return (cnt != 2500);
}
```

Ako vidíte, máme tu jednoduchú slučku. Je to naozaj dobré miesto na začatie glitchovania z dvoch dôvodov:

1. Máme tu naozaj dlhý úsek času s množstvom inštrukcií, ktoré môžeme glitchovať. Na rozdiel od toho v príklade pre Linux sa snažíme zamerať na jedinú inštrukciu.

1. V niektorých scenároch glitchingu hľadáme pomerne špecifický efekt glitchu. Pri tomto jednoduchom výpočte v slučke sa v výsledku prejaví prakticky akákoľvek porucha.

## Modul Glitch

Všetky nastavenia/metódy pre modul glitch sú dostupné pod `scope.glitch`. Ako zvyčajne, dokumentácia k nastaveniam a metódam je dostupná na [ReadtheDocs](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html) alebo pomocou príkazu python `help`:

In [ ]:
help(scope.glitch)

#### Najskôr si prejdeme nastavenia CW Lite/Pro:
* clk_src

> Hodinový signál, ktorý modul glitch DCM používa ako vstup. Možno nastaviť na „target“ alebo „clkgen“. V tomto prípade budeme poskytovať hodiny cieľu, takže toto nastavenie necháme na „clkgen“.

* offset

> Miesto vo výstupnom takte, kde sa má umiestniť glitch. Môže byť v rozsahu `[-48,8, 48,8]`. Často budeme chcieť vyskúšať mnoho offsetov, keď sa pokúšame vyvolať glitch na cieľ.


* width

> Ako široký má byť glitch. Môže byť v rozsahu `[-50, 50]`, hoci nie je dôvod používať šírky < 0. Širšie glitche ľahšie spôsobujú glitche, ale je tiež väčšia pravdepodobnosť, že spôsobia pád cieľa, čo znamená, že pri útoku na cieľ budeme často chcieť vyskúšať celý rad šírok.



* output

> Výstup generovaný modulom glitch. Pri glitchovaní hodín je často najužitočnejšou voľbou clock_xor, pretože táto voľba počas glitchu invertuje hodiny.
* ext_offset

> Počet taktov po spúšťači, po ktorých sa má glitch aplikovať.
* repeat

> Počet taktov, počas ktorých sa glitch opakuje. Vyššie hodnoty zvyšujú počet inštrukcií, ktoré môžu byť glitchované, ale často zvyšujú riziko zlyhania cieľa.

* trigger_src

> Ako spustiť glitch. V tomto tutoriáli chceme automaticky spustiť glitch z triggerového pinu až po aktivácii ChipWhipserera, takže použijeme `ext_single`

Okrem toho budeme musieť ChipWhipserer nariadiť, aby používal výstup modulu glitch ako zdroj hodín pre cieľ, a to nastavením `scope.io.hs2 = „glitch“`. Nastavíme tiež veľké `repeat`, aby bolo glitchovanie jednoduchšie.



## Ovládač glitchov CW

Aby bolo vytváranie glitchových slučiek jednoduchšie, ChipWhisperer obsahuje ovládač glitchov. Začneme inicializáciou s rôznymi možnými výsledkami útoku. Môžete ich definovať podľa vlastného uváženia, ale často stačia tri skupiny:

1. `„success“`, keď mal náš glitch požadovaný účinok
1. `„reset“`, kde mal náš glitch nežiaduci účinok. Často je týmto účinkom zrútenie alebo resetovanie cieľa, preto ho nazývame `„reset“`
1. `„normal“`, kde glitch nemal žiadny badateľný účinok.

Musíme tiež určiť, aké parametre glitchu chceme prehľadávať, v tomto prípade šírku a posun:

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "tries"])

Jednou z výhod ovládača glitch je, že dokáže zobrazovať aktuálne nastavenia. Tieto sa aktualizujú v reálnom čase počas používania ovládača glitch!

In [ ]:
gc.display_stats()

Môžeme tiež vytvoriť graf nastavení, ktorý sa bude aktualizovať v reálnom čase:

In [ ]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None})

`plotdots` je slovník, ktorý špecifikuje ako budeme vykreslovať každú skupinu. V našom príklade zobrazujeme `"success"` pomocou zeleného `+` (`"+g"`), `"reset"` ako červené `x` (`"xr"`), a nebudeme zobrazovať prípady kedy nič mimoriadne nenastane (`None`)


Tento graf automaticky aktualizuje svoje hranice pri pridávaní bodov. Ak chcete špecifikovať hranice osí, môžete tak urobiť nasledovne:

```python
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_bound=(-48, 48), y_bound=(-48, 48))
```

Môžete tiež vybrať, ktoré parametre chcete použiť pre x a y, buď podľa indexu, alebo podľa názvu:

```python
# vymení width a offset 
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index=1, y_index=0)
# alebo
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="offset", y_index="width")

```

Môžeme nastaviť rozmedzie pre obe nastavenia glitchu

In [ ]:
gc.set_range("width", -5, 5)
gc.set_range("offset", -5, 5)

Každé z týchto nastavení sa bude precházdať od min po max na základe hodnoty v "global step"

In [ ]:
gc.set_global_step([5.0, 2.5])

Môžme sa pozrieť aké konkrétne hodnoty to bude prechádzať

In [ ]:
for glitch_setting in gc.glitch_values():
    print("offset: {:4.1f}; width: {:4.1f}".format(glitch_setting[1], glitch_setting[0]))

Začneme s nasledujúcimi nastaveniami. Pri glitchovaní hodinového signálu sa zvyčajne najlepšie osvedčuje použitie „clock_xor“, ktoré vloží glitch v prípade, že je hodinový signál na úrovni „high“ alebo „low“.

In [ ]:
#Basic setup
# set glitch clock
scope.glitch.clk_src = "clkgen" 

scope.glitch.output = "clock_xor" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called

scope.io.hs2 = "glitch"  # output glitch_out on the clock line
print(scope.glitch)

Mali by ste mať všetko potrebné na vytvorenie glitchovej slučky. Pomôžeme vám s úvodom, ale zvyšok je na vás! A ešte pár vecí, na ktoré by ste mali pamätať:

Budete musieť detekovať zlyhania, úspešné glitche a normálne návraty z cieľa. Nebojte sa experimentovať so slučkou: vždy ju môžete reštartovať a spustiť kód znova.
Môžete pokryť väčšiu sadu nastavení glitchov tým, že začnete s veľkými krokmi ovládača glitchov, aby ste získali predstavu, kde sa nachádzajú zaujímavé miesta, a potom zopakujete glitchovú slučku s malými krokmi v zaujímavých oblastiach. Tam, kde je jeden úspešný glitch, je ich pravdepodobne viac!
Svoje glitchovanie môžete podstatne urýchliť tým, že budete zaznamenávať len zlyhania a úspechy, keďže tie sú zvyčajne oveľa vzácnejšie ako normálne správanie v cieli.

In [ ]:
import chipwhisperer.common.results.glitch as glitch
from tqdm.notebook import trange
import struct

scope.glitch.ext_offset = 8


num_tries = 1
gc.set_range("tries", 1, num_tries)

gc.set_range("width", 0, 48)
gc.set_range("offset", -48, 48)
gc.set_global_step([8, 4, 2, 1])

scope.glitch.repeat = 10
gc.set_step("tries", 1)
scope.adc.timeout = 0.1

reboot_flush()

for glitch_setting in gc.glitch_values():
    
    # optional: you can speed up the loop by checking if the trigger never went low
    #           (the target never called trigger_low();) via scope.adc.state
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]

    scope.arm()
    
    target.simpleserial_write("g", bytearray([]))
    
    ret = scope.capture()
    
    val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)#For loop check
    
    # ###################
    # Add your code here
    # ###################
    
    if ret: #here the trigger never went high - sometimes the target is still crashed from a previous glitch
        print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        if invalid_response: # change this to detect an invalid response
            gc.add("reset")
        else:
            # gcnt is the loop counter
            gcnt = struct.unpack("<I", val['payload'])[0]
            
            if gcnt == ???: #normal response
                gc.add("normal")
            else: #glitch!!!
                gc.add("success")

<details>
<summary>Pomôcka 1</summary>

V prípade že loop zbehol bez nejakého glitchu mala by byť návratová hodnota "gcnt = 2500"
takže vieme povedať že ak taká nie je, musel nastať glitch.

</details>

<details>
<summary>Pomôcka 2</summary>

Na detekovanie neplatných hodnôt môžeš použiť:
<pre><code>
    invalid_response = (
            not val['valid'] or
            val['payload'] is None or
            len(val['payload']) != 4
        )
</code></pre>
</details>

### Výsledky

Okrem vykresľovania má ovládač glitchov tiež schopnosť vrátiť výsledky vo forme zoznamu, ktorý zoskupuje parametre a výsledky. Tieto výsledky uvádzajú nielen počet jednotlivých výsledkov, ale aj ich frekvenciu:

In [ ]:
gc.calc()

Taktiež môžme niektoré parametre ignorovať. To spôsobi že sa zoskupia aj ak mali ignorovaný parameter rôzny.

In [ ]:
results = gc.calc(ignore_params="tries")
results

`calc()` nám umožnuje aj triediť podľa parametrov, napr. podľa "success"

In [ ]:
results = gc.calc(ignore_params="tries", sort="success")
results

### Vykresľovanie výsledkov glitchu

Mapu glitchu môžeme znovu vykresliť pomocou metódy `plot_2d()`. Nastavenia sú podobné ako pri metóde `glitch_plot()`. Ak nie je špecifikovaná voľba `plotdots`, použijú sa rovnaké nastavenia ako pri metóde `glitch_plot()`.

#gc.plot_2d(plotdots={"success":"+g", "reset":"xr", "normal":None})

Nezabudnite si tieto nastavenia pre glitching zapísať, pretože ich budeme používať aj v ďalších cvičeniach zameraných na glitching! V skutočnosti budeme vo zvyšných cvičeniach využívať veľkú časť tejto všeobecnej štruktúry kódu, pričom jediné výrazné zmeny budú:

Nezabudnite si tieto nastavenia glitchu zapísať, pretože ich budeme používať aj v ďalších cvičeniach zameraných na glitch! V skutočnosti budeme vo zvyšných cvičeniach využívať veľkú časť tejto všeobecnej štruktúry kódu, pričom jediné výrazné zmeny budú:

### Opakovanie

V tomto laboratóriu sme použili pomerne veľkú hodnotu opakovania (repeat). Ako naznačuje názov, toto nastavenie určuje, koľkokrát sa glitch opakuje (napr. hodnota opakovania 5 umiestni glitche do 5 po sebe idúcich taktovacích cyklov). Zohľadnite, že každý vložený glitch má šancu spôsobiť buď glitch, alebo pád zariadenia. Pre toto cvičenie to bolo dosť výhodné, keďže sme mali veľa rôznych miest, kde sme chceli umiestniť glitch – použitie vysokej hodnoty opakovania zvýšilo našu šancu na pád, ale zároveň zvýšilo našu šancu na úspešný glitch. Pri útoku, kde sa zameriavame na jedinú inštrukciu, v skutočnosti vôbec nezvyšujeme šancu na glitch, ale stále máme zvýšené riziko zlyhania. Ešte horšie je, že úspešný glitch na nesprávnom mieste môže tiež spôsobiť zlyhanie! Z tohto dôvodu je často lepšie použiť nízku hodnotu opakovania pri zameriavaní sa na jedinú inštrukciu.

### Ext Offset

Nastavenie ext offset určuje oneskorenie medzi spustením spúšťača a vložením glitchu. Podobne ako opakovanie (repeat) je založené na celých hodinových cykloch, čo znamená, že hodnota ext offset 10 vloží glitch 10 cyklov po spustení spúšťača. V tomto laboratóriu sme sa o toto nastavenie nemuseli starať, pretože veľká hodnota opakovania nás dostala do požadovanej oblasti. V mnohých aplikáciách to však neplatí a budete musieť vyskúšať glitche s rôznymi hodnotami ext_offset.

### Úspech, reset a normálny stav

Tieto tri výsledné stavy zvyčajne stačia na opísanie väčšiny výsledkov glitchov. To, čo predstavuje úspech, sa však bude líšiť v závislosti od toho, na aký firmware útočíte. Napríklad, ak by sme útočili na autentizáciu v Linuxe, úspech by sme mohli definovať na základe kontroly, či máme alebo nemáme práva root.

In [ ]:
target.dis()
scope.dis()